In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7
!pip install langgraph
!pip install pyngrok
!pip install flask
!pip install request

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cur

In [3]:
from pyngrok import ngrok
# !ps aux | grep ngrok  # Check if ngrok is running
# !kill -9 <2qRXVr2GUcOFKROKoDQx5LZKPEW_7Y943wJ2VMabdjiHWqF8E>
ngrok.set_auth_token("2qh7rqENkqSjp9oWaG1hYxjwgeE_6cTymJfT7YnyoB36tRHiZ")
pub_url=ngrok.connect(5000).public_url

In [4]:
from pyngrok import ngrok
# !ps aux | grep ngrok  # Check if ngrok is running
# !kill -9 <2qRXVr2GUcOFKROKoDQx5LZKPEW_7Y943wJ2VMabdjiHWqF8E>
ngrok.set_auth_token("2qRYoNQDlzFBpMXKqEukQ897U0m_4rGUYYKrXL7Fx43AZPpmH")
pub_url=ngrok.connect(5000).public_url


import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer


from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage

import locale
locale.getpreferredencoding = lambda: "UTF-8"

from flask import Flask, request, jsonify
from pyngrok import ngrok


from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

flask_app = Flask(__name__)
model_name='NousResearch/Llama-2-7b-chat-hf'
device_map = {"": 0}
# Reload model in FP16 and merge it with LoRA weights
new_model_local_url="/content/drive/MyDrive/workspace/data/Llama-2-7b-chat-finetune"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, new_model_local_url)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


#///////////////////////////////////////////////////////////////////////////////////////////////



his=''

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=400)

# Define the function that calls the model
def call_model(state: MessagesState):
    global his
    print("query:",state["messages"][-1].content)
    # prompt = his + state["messages"][-1].content
    prompt=state["messages"][-1].content
    his+= state["messages"][-1].content
    result = pipe(f"<s>[INST] {prompt} [/INST]")
    response = result[0]['generated_text']
    response_1 = response.split('[/INST]')[-1]  # Take text after '[INST]'
    his+= response_1+" "
    print("/n-----------------------------")
    print(his)
    print("/n--------------------------------")

    return {"messages": response_1}


# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "abc123"}}


#///////////////////////////////////////////////////////////////////////////////////////////////

# ngrok.set_auth_token("2qRXVr2GUcOFKROKoDQx5LZKPEW_7Y943wJ2VMabdjiHWqF8E")
# pub_url=ngrok.connect(5000).public_url
print(pub_url)
@flask_app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    input_text = data.get("text", "")
    print("+=============================>",input_text)
    input_messages = [HumanMessage(input_text)]
    output = app.invoke({"messages": input_messages}, config)
    print("+=============================>",output["messages"][-1])
    print("+=============================>",output["messages"][-1].content)

    return jsonify({"response": output["messages"][-1].content})
# Tạo đường hầm ngrok
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

flask_app.run(port=5000)


# Empty VRAM
del model
del pipe
import gc
gc.collect()
gc.collect()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

...

INFO:werkzeug:127.0.0.1 - - [25/Dec/2024 05:35:07] "POST /generate HTTP/1.1" 200 -


/n-----------------------------
What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ.  Which genres in Black Doves are related to drama and mystery? The genres related to drama and mystery are Drama and Mystery.Љ.  What is the rating of the movie Black Doves? AND What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. [1]

The rating of the movie Black Doves is 7.5. [2] What is the rating of the movie Black Doves?. And What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. [1]

The rating of the movie Black Doves is 7.8. [2] The rating of the movie Black Doves is 7.Љ The rating of the movie Black Doves is 7.Љ.

The movie Black Doves was released in 2024 and directed by Alex Gabassi. It belongs to the genres Spy, Thriller, and Mystery. The movie stars Keira Knightley, Ben Whishaw, and Sarah Lancashi

INFO:werkzeug:127.0.0.1 - - [25/Dec/2024 05:35:23] "POST /generate HTTP/1.1" 200 -


/n-----------------------------
What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ.  Which genres in Black Doves are related to drama and mystery? The genres related to drama and mystery are Drama and Mystery.Љ.  What is the rating of the movie Black Doves? AND What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. [1]

The rating of the movie Black Doves is 7.5. [2] What is the rating of the movie Black Doves?. And What is the rating of the movie Black Doves? The rating of the movie Black Doves is 7.Љ. [1]

The rating of the movie Black Doves is 7.8. [2] The rating of the movie Black Doves is 7.Љ The rating of the movie Black Doves is 7.Љ.

The movie Black Doves was released in 2024 and directed by Alex Gabassi. It belongs to the genres Spy, Thriller, and Mystery. The movie stars Keira Knightley, Ben Whishaw, and Sarah Lancashi

0